## Banking BI Data Model — Star Schema & Analytical SQL
*** Ayebazibwe Clare Mujuni | Banking Data Analyst Portfolio**

Builds a complete BI Information System for a fictional bank:
star schema design, ETL pipeline to populate it, and five advanced
analytical SQL queries demonstrating the techniques used in
commercial banking data teams.

**Stack:** DuckDB · Python · Pandas · SQL (CTEs, window functions, RANK, LAG)

In [9]:
import os
from pathlib import Path
os.chdir(Path().resolve())

import duckdb
import pandas as pd
import numpy as np
from datetime import datetime, date, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")
print("DuckDB version:", duckdb.__version__)

Libraries loaded ✓
DuckDB version: 1.5.4


## 1. Star Schema Design (DDL)

The star schema separates data into two types of tables:

- **Fact table** (`fact_transactions`) — stores measurable events:
  transaction amounts, one row per transaction. This is the centre of the star.
- **Dimension tables** — store descriptive context: who, what, where, when.
  Branch, product, customer, date. These are the points of the star.

This structure makes analytical queries fast. Filtering by branch, product,
or time period becomes a simple JOIN rather than a complex subquery across
a normalised transactional schema.

**DuckDB** is used as the analytical database — an embedded engine optimised
for OLAP workloads, equivalent in concept to Redshift or BigQuery but
running locally without a server.

In [10]:
# Connect to DuckDB (creates banking_bi.duckdb file)
conn = duckdb.connect("banking_bi.duckdb")

# ── Drop tables if they exist (for clean reruns) ──────────
conn.execute("DROP TABLE IF EXISTS fact_transactions")
conn.execute("DROP TABLE IF EXISTS dim_account")
conn.execute("DROP TABLE IF EXISTS dim_customer")
conn.execute("DROP TABLE IF EXISTS dim_product")
conn.execute("DROP TABLE IF EXISTS dim_branch")
conn.execute("DROP TABLE IF EXISTS dim_date")

# ── Dimension: Date ───────────────────────────────────────
conn.execute("""
    CREATE TABLE dim_date (
        date_key        INTEGER PRIMARY KEY,  -- YYYYMMDD
        full_date       DATE,
        year            INTEGER,
        quarter         INTEGER,
        month           INTEGER,
        month_name      VARCHAR,
        week            INTEGER,
        day_of_week     VARCHAR,
        is_weekend      BOOLEAN
    )
""")

# ── Dimension: Branch ─────────────────────────────────────
conn.execute("""
    CREATE TABLE dim_branch (
        branch_key      INTEGER PRIMARY KEY,
        branch_name     VARCHAR,
        region          VARCHAR,
        branch_type     VARCHAR,
        opened_date     DATE
    )
""")

# ── Dimension: Product ────────────────────────────────────
conn.execute("""
    CREATE TABLE dim_product (
        product_key     INTEGER PRIMARY KEY,
        product_name    VARCHAR,
        product_type    VARCHAR,
        interest_rate   DECIMAL(5,2),
        is_active       BOOLEAN
    )
""")

# ── Dimension: Customer ───────────────────────────────────
conn.execute("""
    CREATE TABLE dim_customer (
        customer_key    INTEGER PRIMARY KEY,
        customer_id     VARCHAR,
        customer_name   VARCHAR,
        segment         VARCHAR,
        gender          VARCHAR,
        age_band        VARCHAR,
        join_date       DATE
    )
""")

# ── Dimension: Account ────────────────────────────────────
conn.execute("""
    CREATE TABLE dim_account (
        account_key     INTEGER PRIMARY KEY,
        account_number  VARCHAR,
        customer_key    INTEGER,
        product_key     INTEGER,
        branch_key      INTEGER,
        open_date       DATE,
        status          VARCHAR
    )
""")

# ── Fact: Transactions ────────────────────────────────────
conn.execute("""
    CREATE TABLE fact_transactions (
        transaction_id  INTEGER PRIMARY KEY,
        date_key        INTEGER,
        account_key     INTEGER,
        branch_key      INTEGER,
        product_key     INTEGER,
        txn_type        VARCHAR,
        amount          DECIMAL(18,2),
        balance_after   DECIMAL(18,2)
    )
""")

print("Star schema created ✓")
print()
print(conn.execute("SHOW TABLES").df().to_string(index=False))

Star schema created ✓

             name
      dim_account
       dim_branch
     dim_customer
         dim_date
      dim_product
fact_transactions


## 2. ETL — Load Dimension Tables

### dim_date
The date dimension pre-computes every calendar attribute for each day
(year, quarter, month, week, day of week, is_weekend). This allows
filtering by "Q3" or "weekdays only" with a simple WHERE clause instead
of calculating date parts in every query.

1,096 rows — one per day from 2022-01-01 to 2024-12-31.

In [11]:
# Generate one row per day for 3 years (2022–2024)
dates = pd.date_range(start='2022-01-01', end='2024-12-31', freq='D')

month_names = {1:'January',2:'February',3:'March',4:'April',
               5:'May',6:'June',7:'July',8:'August',
               9:'September',10:'October',11:'November',12:'December'}

days_of_week = {0:'Monday',1:'Tuesday',2:'Wednesday',
                3:'Thursday',4:'Friday',5:'Saturday',6:'Sunday'}

# Note: DataFrame named dim_date_df to avoid conflict with the table name
dim_date_df = pd.DataFrame({
    'date_key':    [int(d.strftime('%Y%m%d')) for d in dates],
    'full_date':   dates,
    'year':        dates.year,
    'quarter':     dates.quarter,
    'month':       dates.month,
    'month_name':  [month_names[m] for m in dates.month],
    'week':        dates.isocalendar().week.astype(int),
    'day_of_week': [days_of_week[d] for d in dates.dayofweek],
    'is_weekend':  dates.dayofweek >= 5
})

conn.execute("DELETE FROM dim_date")  # clear any bad rows
conn.execute("INSERT INTO dim_date SELECT * FROM dim_date_df")
count = conn.execute("SELECT COUNT(*) FROM dim_date").fetchone()[0]
print(f"dim_date loaded: {count:,} rows (one per day 2022–2024)")

dim_date loaded: 1,096 rows (one per day 2022–2024)


### dim_branch, dim_product, dim_customer

- **6 branches** across Central, Eastern and Western Uganda regions
- **6 products** across two types: Loans (Personal, Business, Mortgage)
  and Deposits (Savings, Fixed Deposit, Current Account)
- **500 customers** with segment, gender, age band and join date

In production these tables are loaded from the core banking system's
customer master file and product catalogue.

In [12]:
random.seed(42)
np.random.seed(42)

# ── dim_branch ────────────────────────────────────────────
branches = pd.DataFrame({
    'branch_key':  [1, 2, 3, 4, 5, 6],
    'branch_name': ['Kampala Main', 'Nakawa', 'Ntinda', 
                    'Entebbe', 'Jinja', 'Mbarara'],
    'region':      ['Central', 'Central', 'Central',
                    'Central', 'Eastern', 'Western'],
    'branch_type': ['HQ', 'Branch', 'Branch',
                    'Branch', 'Branch', 'Branch'],
    'opened_date': pd.to_datetime(['2010-01-01','2012-06-01','2013-03-01',
                                   '2014-09-01','2015-01-01','2016-07-01'])
})

conn.execute("INSERT INTO dim_branch SELECT * FROM branches")
print(f"dim_branch loaded: {len(branches)} rows")

# ── dim_product ───────────────────────────────────────────
products = pd.DataFrame({
    'product_key':   [1, 2, 3, 4, 5, 6],
    'product_name':  ['Personal Loan', 'Business Loan', 'Mortgage',
                      'Savings Account', 'Fixed Deposit', 'Current Account'],
    'product_type':  ['Loan', 'Loan', 'Loan',
                      'Deposit', 'Deposit', 'Deposit'],
    'interest_rate': [18.5, 21.0, 16.0, 7.5, 11.0, 0.0],
    'is_active':     [True] * 6
})

conn.execute("INSERT INTO dim_product SELECT * FROM products")
print(f"dim_product loaded: {len(products)} rows")

# ── dim_customer ──────────────────────────────────────────
n_customers = 500
segments  = ['Champion', 'Loyal', 'Promising', 'At Risk', 'Lost']
genders   = ['Male', 'Female']
age_bands = ['18-25', '26-35', '36-45', '46-55', '55+']

customers = pd.DataFrame({
    'customer_key':  range(1, n_customers + 1),
    'customer_id':   [f'CUST-{str(i).zfill(4)}' for i in range(1, n_customers + 1)],
    'customer_name': [f'Customer {i}' for i in range(1, n_customers + 1)],
    'segment':       random.choices(segments, k=n_customers),
    'gender':        random.choices(genders, k=n_customers),
    'age_band':      random.choices(age_bands, k=n_customers),
    'join_date':     pd.to_datetime([
                         date(2022,1,1) + timedelta(days=random.randint(0, 1095))
                         for _ in range(n_customers)
                     ])
})

conn.execute("INSERT INTO dim_customer SELECT * FROM customers")
print(f"dim_customer loaded: {len(customers)} rows")

dim_branch loaded: 6 rows
dim_product loaded: 6 rows
dim_customer loaded: 500 rows


### dim_account & fact_transactions

Each customer holds 1–3 accounts across different products and branches —
reflecting real banking behaviour where customers cross-hold products.

**20,000 transactions** loaded into the fact table across 3 years.
Each fact row references four dimension tables via foreign keys:
date_key, account_key, branch_key, product_key.

This is the ETL (Extract, Transform, Load) step — in a production bank,
this runs overnight pulling from the core banking system into the BI layer.

In [13]:
# ── dim_account ───────────────────────────────────────────
# Each customer has 1-3 accounts across different products and branches
account_rows = []
account_key = 1

for cust_key in range(1, n_customers + 1):
    n_accounts = random.randint(1, 3)
    for _ in range(n_accounts):
        account_rows.append({
            'account_key':    account_key,
            'account_number': f'ACC-{str(account_key).zfill(6)}',
            'customer_key':   cust_key,
            'product_key':    random.randint(1, 6),
            'branch_key':     random.randint(1, 6),
            'open_date':      date(2022,1,1) + timedelta(days=random.randint(0, 1095)),
            'status':         random.choices(['Active','Dormant','Closed'],
                                            weights=[80, 15, 5])[0]
        })
        account_key += 1

accounts = pd.DataFrame(account_rows)
accounts['open_date'] = pd.to_datetime(accounts['open_date'])
conn.execute("INSERT INTO dim_account SELECT * FROM accounts")
print(f"dim_account loaded: {len(accounts)} accounts")

# ── fact_transactions ─────────────────────────────────────
n_txns = 20000
all_dates = pd.date_range('2022-01-01', '2024-12-31')
txn_types = ['Deposit', 'Withdrawal', 'Loan Disbursement',
             'Loan Repayment', 'Transfer In', 'Transfer Out']

fact = pd.DataFrame({
    'transaction_id': range(1, n_txns + 1),
    'full_date':      np.random.choice(all_dates, size=n_txns),
    'account_key':    np.random.randint(1, len(accounts) + 1, n_txns),
    'txn_type':       random.choices(txn_types, k=n_txns),
    'amount':         np.round(np.random.exponential(scale=2_000_000, size=n_txns), 0)
})

# Derive keys from account table via merge
fact = fact.merge(
    accounts[['account_key', 'branch_key', 'product_key']],
    on='account_key', how='left'
)

fact['date_key']      = fact['full_date'].dt.strftime('%Y%m%d').astype(int)
fact['balance_after'] = np.round(np.random.uniform(100_000, 50_000_000, n_txns), 0)

fact_final = fact[['transaction_id', 'date_key', 'account_key',
                    'branch_key', 'product_key', 'txn_type',
                    'amount', 'balance_after']]

conn.execute("INSERT INTO fact_transactions SELECT * FROM fact_final")
print(f"fact_transactions loaded: {len(fact_final):,} rows")

dim_account loaded: 1023 accounts


fact_transactions loaded: 20,000 rows


## 3. Schema Verification

Confirm all tables loaded correctly and test a multi-table join
across the full star schema — the foundation of every analytical
query that follows.

In [14]:
print("ROW COUNTS PER TABLE")
print("=" * 35)
for table in ['dim_date', 'dim_branch', 'dim_product',
              'dim_customer', 'dim_account', 'fact_transactions']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<25} {count:>7,}")

print()
print("Sample fact join (5 rows):")
sample = conn.execute("""
    SELECT 
        f.transaction_id,
        d.full_date,
        b.branch_name,
        p.product_name,
        f.txn_type,
        f.amount
    FROM fact_transactions f
    JOIN dim_date    d ON f.date_key    = d.date_key
    JOIN dim_branch  b ON f.branch_key  = b.branch_key
    JOIN dim_product p ON f.product_key = p.product_key
    LIMIT 5
""").df()
print(sample.to_string(index=False))

ROW COUNTS PER TABLE
  dim_date                    1,096
  dim_branch                      6
  dim_product                     6
  dim_customer                  500
  dim_account                 1,023
  fact_transactions          20,000

Sample fact join (5 rows):
 transaction_id  full_date  branch_name    product_name          txn_type    amount
              1 2024-05-10       Ntinda   Business Loan Loan Disbursement 5225781.0
              2 2024-12-31 Kampala Main Current Account           Deposit 8530761.0
              3 2024-11-10      Entebbe   Business Loan      Transfer Out 2732477.0
              4 2022-05-02       Nakawa   Fixed Deposit       Transfer In 4048745.0
              5 2023-04-12      Mbarara Savings Account    Loan Repayment 2240069.0


## 4. Analytical SQL Queries

### Query 1 — Monthly Commercial KPIs

Aggregates transaction volume by month with a split between loan
and deposit products using CASE WHEN inside SUM — a standard
banking reporting pattern for the monthly commercial scorecard.

**SQL techniques:** GROUP BY, SUM with CASE WHEN, date dimension filtering.

In [15]:
print("QUERY 1: Monthly Commercial KPIs")
print("=" * 60)

q1 = conn.execute("""
    SELECT
        d.year,
        d.month,
        d.month_name,
        COUNT(DISTINCT f.account_key)          AS active_accounts,
        COUNT(f.transaction_id)                AS total_transactions,
        ROUND(SUM(f.amount) / 1e6, 2)         AS total_volume_ugx_m,
        ROUND(AVG(f.amount) / 1e6, 3)         AS avg_txn_size_ugx_m,
        ROUND(SUM(CASE WHEN p.product_type = 'Loan'
                   THEN f.amount ELSE 0 END) / 1e6, 2)  AS loan_volume_ugx_m,
        ROUND(SUM(CASE WHEN p.product_type = 'Deposit'
                   THEN f.amount ELSE 0 END) / 1e6, 2)  AS deposit_volume_ugx_m
    FROM fact_transactions f
    JOIN dim_date    d ON f.date_key    = d.date_key
    JOIN dim_product p ON f.product_key = p.product_key
    GROUP BY d.year, d.month, d.month_name
    ORDER BY d.year, d.month
""").df()

print(q1.to_string(index=False))
q1.to_csv("query1_monthly_kpis.csv", index=False)
print("\nSaved ✓")

QUERY 1: Monthly Commercial KPIs
 year  month month_name  active_accounts  total_transactions  total_volume_ugx_m  avg_txn_size_ugx_m  loan_volume_ugx_m  deposit_volume_ugx_m
 2022      1    January              445                 573             1188.82               2.075             575.78                613.04
 2022      2   February              371                 484              930.12               1.922             440.12                490.00
 2022      3      March              423                 553             1058.19               1.914             508.86                549.34
 2022      4      April              403                 543             1056.49               1.946             442.69                613.80
 2022      5        May              422                 553             1027.10               1.857             525.07                502.02
 2022      6       June              407                 537             1062.71               1.979             53

### Query 2 — Branch Performance Ranking

Ranks branches by transaction volume each year and calculates
year-on-year change using LAG — showing which branches grew or
contracted relative to prior year.

**SQL techniques:** CTE (WITH clause), RANK() window function,
LAG() window function with PARTITION BY and ORDER BY.

In [16]:
print("QUERY 2: Branch Performance Ranking — Window Functions")
print("=" * 60)

q2 = conn.execute("""
    WITH branch_annual AS (
        SELECT
            d.year,
            b.branch_name,
            b.region,
            COUNT(f.transaction_id)         AS total_txns,
            ROUND(SUM(f.amount) / 1e6, 2)  AS total_volume_ugx_m,
            COUNT(DISTINCT f.account_key)   AS active_accounts
        FROM fact_transactions f
        JOIN dim_date   d ON f.date_key   = d.date_key
        JOIN dim_branch b ON f.branch_key = b.branch_key
        GROUP BY d.year, b.branch_name, b.region
    )
    SELECT
        year,
        branch_name,
        region,
        total_txns,
        total_volume_ugx_m,
        active_accounts,
        RANK() OVER (PARTITION BY year ORDER BY total_volume_ugx_m DESC)
            AS volume_rank,
        ROUND(total_volume_ugx_m - LAG(total_volume_ugx_m)
              OVER (PARTITION BY branch_name ORDER BY year), 2)
            AS yoy_change_ugx_m
    FROM branch_annual
    ORDER BY year, volume_rank
""").df()

print(q2.to_string(index=False))
q2.to_csv("query2_branch_ranking.csv", index=False)
print("\nSaved ✓")

QUERY 2: Branch Performance Ranking — Window Functions
 year  branch_name  region  total_txns  total_volume_ugx_m  active_accounts  volume_rank  yoy_change_ugx_m
 2022      Entebbe Central        1101             2356.95              163            1               NaN
 2022      Mbarara Western        1184             2323.61              182            2               NaN
 2022 Kampala Main Central        1133             2227.81              184            3               NaN
 2022       Nakawa Central        1056             2064.05              169            4               NaN
 2022       Ntinda Central        1037             1974.42              166            5               NaN
 2022        Jinja Eastern         969             1850.75              156            6               NaN
 2023       Ntinda Central        1151             2467.21              167            1            492.79
 2023 Kampala Main Central        1207             2423.34              184            2 

### Query 3 — Product Penetration by Customer Segment

Measures how many products each customer holds (cross-sell depth)
and total spend, grouped by commercial segment.

**Key finding:** "Lost" customers have the highest total segment value
(UGX 9.1B) — they were not low-value customers, they were
poorly retained. Strong business case for a targeted win-back campaign.

**SQL techniques:** Multi-level CTE, COUNT DISTINCT, AVG aggregation.

In [17]:
print("QUERY 3: Product Penetration by Customer Segment — CTE")
print("=" * 60)

q3 = conn.execute("""
    WITH customer_products AS (
        SELECT
            c.customer_key,
            c.segment,
            COUNT(DISTINCT a.product_key)       AS products_held,
            ROUND(SUM(f.amount) / 1e6, 2)       AS total_spend_ugx_m
        FROM dim_customer c
        JOIN dim_account     a ON c.customer_key = a.customer_key
        JOIN fact_transactions f ON a.account_key = f.account_key
        GROUP BY c.customer_key, c.segment
    )
    SELECT
        segment,
        COUNT(customer_key)                         AS customers,
        ROUND(AVG(products_held), 2)                AS avg_products_held,
        MAX(products_held)                          AS max_products_held,
        ROUND(AVG(total_spend_ugx_m), 2)            AS avg_spend_ugx_m,
        ROUND(SUM(total_spend_ugx_m), 2)            AS total_segment_value_ugx_m
    FROM customer_products
    GROUP BY segment
    ORDER BY total_segment_value_ugx_m DESC
""").df()

print(q3.to_string(index=False))
q3.to_csv("query3_product_penetration.csv", index=False)
print("\nSaved ✓")

QUERY 3: Product Penetration by Customer Segment — CTE
  segment  customers  avg_products_held  max_products_held  avg_spend_ugx_m  total_segment_value_ugx_m
     Lost        107               1.92                  3            85.04                    9099.69
    Loyal        110               1.79                  3            77.77                    8554.74
  At Risk         95               1.88                  3            84.43                    8020.98
Promising         99               1.81                  3            73.47                    7273.61
 Champion         89               1.85                  3            80.16                    7133.80

Saved ✓


### Query 4 — Month-on-Month Volume Growth

Calculates the absolute and percentage change in transaction volume
month by month across the full 3-year period using LAG.

**Key finding:** December 2024 shows +23.1% MoM growth — driven by
year-end salary payments, business settlements, and seasonal spending.
A consistent pattern across Ugandan commercial banks.

**SQL techniques:** LAG() window function, derived percentage calculation,
ordering across multiple year/month columns.

In [18]:
print("QUERY 4: Month-on-Month Volume Growth — LAG Function")
print("=" * 60)

q4 = conn.execute("""
    WITH monthly_volume AS (
        SELECT
            d.year,
            d.month,
            d.month_name,
            ROUND(SUM(f.amount) / 1e6, 2) AS volume_ugx_m
        FROM fact_transactions f
        JOIN dim_date d ON f.date_key = d.date_key
        GROUP BY d.year, d.month, d.month_name
    )
    SELECT
        year,
        month,
        month_name,
        volume_ugx_m,
        LAG(volume_ugx_m) OVER (ORDER BY year, month)   AS prev_month_ugx_m,
        ROUND(volume_ugx_m - LAG(volume_ugx_m)
              OVER (ORDER BY year, month), 2)            AS mom_change_ugx_m,
        ROUND((volume_ugx_m - LAG(volume_ugx_m)
               OVER (ORDER BY year, month))
               / LAG(volume_ugx_m)
               OVER (ORDER BY year, month) * 100, 1)    AS mom_growth_pct
    FROM monthly_volume
    ORDER BY year, month
""").df()

print(q4.to_string(index=False))
q4.to_csv("query4_mom_growth.csv", index=False)
print("\nSaved ✓")

QUERY 4: Month-on-Month Volume Growth — LAG Function
 year  month month_name  volume_ugx_m  prev_month_ugx_m  mom_change_ugx_m  mom_growth_pct
 2022      1    January       1188.82               NaN               NaN             NaN
 2022      2   February        930.12           1188.82           -258.70           -21.8
 2022      3      March       1058.19            930.12            128.07            13.8
 2022      4      April       1056.49           1058.19             -1.70            -0.2
 2022      5        May       1027.10           1056.49            -29.39            -2.8
 2022      6       June       1062.71           1027.10             35.61             3.5
 2022      7       July       1044.12           1062.71            -18.59            -1.7
 2022      8     August       1195.40           1044.12            151.28            14.5
 2022      9  September       1065.74           1195.40           -129.66           -10.8
 2022     10    October       1109.13          

### Query 5 — Top 3 Customers by Volume per Branch

Identifies the three highest-value customers at each branch using
RANK with PARTITION BY — the query a relationship manager would
run to prioritise their customer calls.

**Key finding:** Kampala Main's top 3 customers are all flagged "At Risk"
with volumes between UGX 124M and UGX 163M. This is an immediate
commercial alert — losing these three customers represents significant
revenue exposure at the bank's flagship branch.

**SQL techniques:** Two-level CTE, RANK() with PARTITION BY branch,
three-table JOIN across fact and dimension tables.

In [19]:
print("QUERY 5: Top 3 Customers by Volume per Branch")
print("=" * 60)

q5 = conn.execute("""
    WITH customer_branch_volume AS (
        SELECT
            b.branch_name,
            c.customer_id,
            c.segment,
            ROUND(SUM(f.amount) / 1e6, 2)      AS total_volume_ugx_m,
            COUNT(f.transaction_id)             AS txn_count
        FROM fact_transactions f
        JOIN dim_account  a ON f.account_key  = a.account_key
        JOIN dim_customer c ON a.customer_key = c.customer_key
        JOIN dim_branch   b ON f.branch_key   = b.branch_key
        GROUP BY b.branch_name, c.customer_id, c.segment
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (
                PARTITION BY branch_name
                ORDER BY total_volume_ugx_m DESC
            ) AS branch_rank
        FROM customer_branch_volume
    )
    SELECT
        branch_name,
        branch_rank,
        customer_id,
        segment,
        total_volume_ugx_m,
        txn_count
    FROM ranked
    WHERE branch_rank <= 3
    ORDER BY branch_name, branch_rank
""").df()

print(q5.to_string(index=False))
q5.to_csv("query5_top_customers_by_branch.csv", index=False)
print("\nSaved ✓")

QUERY 5: Top 3 Customers by Volume per Branch
 branch_name  branch_rank customer_id   segment  total_volume_ugx_m  txn_count
     Entebbe            1   CUST-0392   At Risk              121.02         43
     Entebbe            2   CUST-0049   At Risk               94.82         43
     Entebbe            3   CUST-0113 Promising               92.58         39
       Jinja            1   CUST-0433  Champion              100.78         49
       Jinja            2   CUST-0219  Champion               97.80         44
       Jinja            3   CUST-0181   At Risk               95.43         50
Kampala Main            1   CUST-0457   At Risk              163.24         70
Kampala Main            2   CUST-0103   At Risk              126.57         48
Kampala Main            3   CUST-0488   At Risk              124.37         71
     Mbarara            1   CUST-0194      Lost              128.92         51
     Mbarara            2   CUST-0177  Champion              106.87         65
     M

In [20]:
# Save all query results together for the dashboard
summary = {
    'total_transactions': int(conn.execute("SELECT COUNT(*) FROM fact_transactions").fetchone()[0]),
    'total_volume_ugx_b': round(conn.execute("SELECT SUM(amount)/1e9 FROM fact_transactions").fetchone()[0], 2),
    'active_accounts':    int(conn.execute("SELECT COUNT(*) FROM dim_account WHERE status='Active'").fetchone()[0]),
    'total_customers':    int(conn.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]),
    'total_branches':     int(conn.execute("SELECT COUNT(*) FROM dim_branch").fetchone()[0]),
}

print("PORTFOLIO SUMMARY")
print("=" * 40)
for k, v in summary.items():
    print(f"  {k:<28} {v:>10}")

conn.close()
print("\nDatabase connection closed ✓")
print("All query outputs saved as CSV ✓")

PORTFOLIO SUMMARY
  total_transactions                20000
  total_volume_ugx_b                40.08
  active_accounts                     820
  total_customers                     500
  total_branches                        6

Database connection closed ✓
All query outputs saved as CSV ✓


In [22]:

conn = duckdb.connect("banking_bi.duckdb")

summary = {
    'total_transactions': int(conn.execute("SELECT COUNT(*) FROM fact_transactions").fetchone()[0]),
    'total_volume_ugx_b': round(conn.execute("SELECT SUM(amount)/1e9 FROM fact_transactions").fetchone()[0], 2),
    'active_accounts':    int(conn.execute("SELECT COUNT(*) FROM dim_account WHERE status='Active'").fetchone()[0]),
    'total_customers':    int(conn.execute("SELECT COUNT(*) FROM dim_customer").fetchone()[0]),
    'total_branches':     int(conn.execute("SELECT COUNT(*) FROM dim_branch").fetchone()[0]),
}

print("PORTFOLIO SUMMARY")
print("=" * 40)
for k, v in summary.items():
    print(f"  {k:<28} {v:>10}")

conn.close()
print("\nDatabase connection closed ✓")
print("All query outputs saved as CSV ✓")

PORTFOLIO SUMMARY
  total_transactions                20000
  total_volume_ugx_b                40.08
  active_accounts                     820
  total_customers                     500
  total_branches                        6

Database connection closed ✓
All query outputs saved as CSV ✓
